# 05 — Lancement du modèle (inférence sortie + télé)

**Objectif :** charger les bundles déployés, scorer un séjour du split **test** avec `score_single` pour les deux scores.

> Prérequis : `models/score_*_bundle.joblib` (notebook 04 ou training déjà présent).  
> Suite : `06_reponse_modele.ipynb`


## 1. Imports & chemins


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "patients.csv").is_file() and (p / "Sujet.md").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

from src.data.paths import ProjectPaths
from notebooks._utils.schema import FEATURES_CLES, SCHEMA_ATTENDU

paths = ProjectPaths(root=ROOT)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
print("Racine :", ROOT)
print("Curated :", paths.curated)
print("Models  :", paths.models)


## 2. Chargement bundles + features test


In [ ]:
import json
import joblib

from src.model.inference import score_single
from src.model.prepare import split_xy

bundle_sortie = joblib.load(paths.models / "score_sortie_bundle.joblib")
bundle_tele = joblib.load(paths.models / "score_tele_bundle.joblib")
metrics_sortie = json.loads((paths.models / "score_sortie_metrics.json").read_text(encoding="utf-8"))
metrics_tele = json.loads((paths.models / "score_tele_metrics.json").read_text(encoding="utf-8"))

features_sortie = pd.read_parquet(paths.curated / "features_score_sortie.parquet")
features_tele = pd.read_parquet(paths.curated / "features_score_tele.parquet")

print("retenu sortie :", metrics_sortie.get("retenu"))
print("retenu télé   :", metrics_tele.get("retenu"))


## 3. Sélection d’un séjour test


In [ ]:
# Premier séjour du split test (reproductible)
candidats = features_sortie.loc[features_sortie["split"] == "test", "SejourID"]
if candidats.empty:
    raise ValueError("Aucun séjour dans le split test.")
SEJOUR_ID = str(candidats.iloc[0])
print("SejourID retenu :", SEJOUR_ID)

row_sortie = features_sortie.loc[features_sortie["SejourID"] == SEJOUR_ID]
row_tele = features_tele.loc[features_tele["SejourID"] == SEJOUR_ID]
if row_sortie.empty or row_tele.empty:
    raise KeyError(f"SejourID introuvable dans les features : {SEJOUR_ID}")

X_so, _, _ = split_xy(row_sortie)
X_te, _, _ = split_xy(row_tele)
display(row_sortie[["SejourID", "PatientID", "Readmission30j", "split", "DureeSejour", "Service"]])


## 4. score_single — sortie et télé


In [ ]:
score_so = score_single(bundle_sortie, metrics_sortie, X_so)
score_te = score_single(bundle_tele, metrics_tele, X_te)

comparatif = pd.DataFrame([
    {
        "score": "sortie",
        "SejourID": SEJOUR_ID,
        "modele": score_so["modele"],
        "proba": round(score_so["proba"], 4),
        "seuil": round(score_so["seuil"], 4),
        "alerte": score_so["alerte"],
    },
    {
        "score": "tele",
        "SejourID": SEJOUR_ID,
        "modele": score_te["modele"],
        "proba": round(score_te["proba"], 4),
        "seuil": round(score_te["seuil"], 4),
        "alerte": score_te["alerte"],
    },
])
display(comparatif)

# Conservé pour le notebook réponse
RESULTATS_LANCEMENT = {
    "SejourID": SEJOUR_ID,
    "sortie": score_so,
    "tele": score_te,
    "X_sortie": X_so,
    "X_tele": X_te,
}
print("✓ Inférence OK — enchaînez avec 06_reponse_modele.ipynb (même SejourID)")
print("SEJOUR_ID =", SEJOUR_ID)
